In [0]:
# %pip install ax-platform
%pip install xgboost
%restart_python

In [0]:
import importlib
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, PredictionErrorDisplay

from xgboost import XGBRegressor, plot_importance

# from ax.service.managed_loop import optimize
# from ax.api.client import Client
# from ax.api.configs import RangeParameterConfig

In [0]:
df = spark.sql('''
SELECT * 
FROM workspace.`site-speed-recommendation`.combined
WHERE performance_score is not null
''').toPandas()

In [0]:
df_copy = df.copy()

high_missing_col = ['mainthread_garbageCollection','EXPERIMENTAL_TIME_TO_FIRST_BYTE','INTERACTION_TO_NEXT_PAINT']
df_copy = df_copy.drop(columns=high_missing_col)
df_copy=df_copy.dropna()

y = df_copy['largest-contentful-paint']

drop_cols = ['domain', 'url', 'performance_score', 'largest-contentful-paint','cumulative-layout-shift','first-contentful-paint','total-blocking-time','speed-index', 'interactive']
df_copy = df_copy.drop(columns=drop_cols)


In [0]:
X_encoded = pd.get_dummies(
    df_copy,
    columns=["strategy"],
    drop_first=False
)
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [0]:
client3 = Client()

In [0]:
client3.configure_experiment(
    parameters=[
        RangeParameterConfig(
            name="max_depth",
            parameter_type="int",
            bounds=(3, 15),
        ),
        RangeParameterConfig(
            name="learning_rate",
            parameter_type="float",
            bounds=(0.01, 0.3),
        ),
        RangeParameterConfig(
            name="subsample",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="colsample_bytree",
            parameter_type="float",
            bounds=(0.5, 1.0),
        ),
        RangeParameterConfig(
            name="n_estimators",
            parameter_type="int",
            bounds=(100, 1000),
        ),
        RangeParameterConfig(
            name="min_child_weight",
            parameter_type="int",
            bounds=(1, 20),
        ),
        RangeParameterConfig(
            name="gamma",
            parameter_type="float",
            bounds=(0, 1.0),
        ),
        RangeParameterConfig(
            name="reg_alpha",
            parameter_type="float",
            bounds=(0.0, 5.0),
        ),
        RangeParameterConfig(
            name="reg_lambda",
            parameter_type="float",
            bounds=(0.01, 100.0),
            scaling="log"
        ),
    ]
)
client3.configure_optimization(objective="-mae")

In [0]:
def evaluate_xgb(parameters):
    base_model = XGBRegressor(
        max_depth=int(parameters["max_depth"]),
        learning_rate=parameters["learning_rate"],
        n_estimators=int(parameters["n_estimators"]),
        subsample=parameters["subsample"],
        colsample_bytree=parameters["colsample_bytree"],
        min_child_weight=parameters["min_child_weight"],
        reg_alpha=parameters["reg_alpha"],
        reg_lambda=parameters["reg_lambda"],
        gamma=parameters["gamma"],
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

    model = TransformedTargetRegressor(
        regressor=base_model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1,
    )

    mae = -scores.mean()

    return {"mae": mae}

In [0]:
for i in range(50):
    trails = client3.get_next_trials(max_trials=1)

    for trial_index, parameters in trails.items():
        result = evaluate_xgb(parameters)

        client3.complete_trial(
            trial_index = trial_index,
            raw_data = result,
        )

        print(f"Trial {trial_index}: MAE = {result["mae"]:.4f}")
        print(parameters)

In [0]:
best_parameters = client3.get_best_parameterization()
best_parameters

In [0]:
# {'max_depth': 5,
#   'learning_rate': 0.1705737230262246,
#   'subsample': 1.0,
#   'colsample_bytree': 1.0,
#   'n_estimators': 700,
#   'min_child_weight': 1,
#   'gamma': 0.0,
#   'reg_alpha': 0.0,
#   'reg_lambda': 50.0}

xgb = XGBRegressor(
    max_depth=5,
    learning_rate=0.1705737230262246,
    subsample=1.0,
    colsample_bytree=1,
    n_estimators=700,
    min_child_weight=1,
    gamma=0,
    reg_alpha=0.,
    reg_lambda=50.,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model = TransformedTargetRegressor(
    regressor=xgb,
    func=np.log1p,
    inverse_func=np.expm1
)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

print("XGBoost CV MAE scores:", -scores)
print("Mean CV MAE:", -scores.mean())
print("CV MAE std:", scores.std())

## Get one unseen-fold prediction for every training row
oof_pred = cross_val_predict(
    model,
    X_train,
    y_train,
    cv=cv,
    n_jobs=-1
)

# Margin that contains approximately 90% of past CV errors
absolute_errors = np.abs(y_train - oof_pred)
error_90 = np.quantile(absolute_errors, 0.90)

print(f"90% prediction interval margin: ±{error_90:.2f} seconds")

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

rmse = np.sqrt(mean_squared_error(y_test, test_pred))
r2 = r2_score(y_test, test_pred)

display = PredictionErrorDisplay(y_true=y_test, y_pred=test_pred)
display.plot()

print(f"Train MAE: {train_mae}")
print(f"Test MAE: {test_mae}")
print(f"Test R-squared: {r2}")
print(f"Test RMSE: {rmse}")

In [0]:
feature_columns = X_train.columns.tolist()
feature_columns

In [0]:
fig, ax = plt.subplots(figsize=(8, 5))

plot_importance(
    model.regressor_,
    ax=ax,
    max_num_features=15,
    importance_type="gain"
)

plt.show()

In [0]:
importances = pd.Series(
    model.regressor_.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances)

In [0]:
# {'max_depth': 10,
#   'learning_rate': 0.19143324653710173,
#   'subsample': 1.0,
#   'colsample_bytree': 0.5,
#   'n_estimators': 700,
#   'min_child_weight': 20,
#   'gamma': 0.0,
#   'reg_alpha': 0.0,
#   'reg_lambda': 70.0}

# XGBoost CV MAE scores: [1439.83908757 1469.40377404 1656.88240604 1598.88793592 1480.70475285]
# Mean CV MAE: 1529.1435912827897
# CV MAE std: 83.75194281019867
# Train MAE: 210.44023460729784
# Test MAE: 1343.7251993277532
# Test R-squared: 0.8692862543867691
# Test RMSE: 3272.804311044904

In [0]:
# using this for the final model

# {'max_depth': 5,
#   'learning_rate': 0.1705737230262246,
#   'subsample': 1.0,
#   'colsample_bytree': 1.0,
#   'n_estimators': 700,
#   'min_child_weight': 1,
#   'gamma': 0.0,
#   'reg_alpha': 0.0,
#   'reg_lambda': 50.0}

# XGBoost CV MAE scores: [1494.38997346 1449.70460882 1665.75833705 1618.19436202 1477.07960092]
# Mean CV MAE: 1541.0253764522636
# CV MAE std: 84.99031640812784
# Train MAE: 623.1703938167633
# Test MAE: 1350.1705593890067
# Test R-squared: 0.8607528991764577
# Test RMSE: 3377.944343387083

In [0]:
# USING THIS ONE. THis one has interactive in it

# {'max_depth': 6,
#   'learning_rate': 0.09104239470586255,
#   'subsample': 1.0,
#   'colsample_bytree': 1.0,
#   'n_estimators': 633,
#   'min_child_weight': 15,
#   'gamma': 0.0,
#   'reg_alpha': 0.504961282089077,
#   'reg_lambda': 20.0}

# XGBoost CV MAE scores: [1193.50819305 1202.06297932 1329.7864133  1289.03671206 1134.96709795]
# Mean CV MAE: 1229.8722791340654
# CV MAE std: 70.12269484001024
# Train MAE: 518.8633434174885
# Test MAE: 1082.5541131979107
# Test R-squared: 0.9178032099123331
# Test RMSE: 2595.2956437522603

In [0]:
# %skip
# model.regressor_.save_model("/Volumes/workspace/site-speed-recommendation/models/0001.bin")
joblib.dump(model, "/Volumes/workspace/site-speed-recommendation/models/lcp_model.joblib")